# Week 3, Day 2 — Asset Management Game (6 stocks)

A reproducible reinforcement-learning market: **06 Jul–04 Aug**, six synthetic stocks, and a player who must always own exactly two stocks. The goal is to maximize profit.

The price path is synthetic, so this is an RL lesson—not financial advice.

## MDP design

- **State** = `(trading day, current two-stock portfolio)`. With a fixed market path, these determine the next transition.
- **Action** = one of the 15 unordered pairs from six stocks. This guarantees the agent always holds exactly two stocks.
- **Transition** = rebalance to the chosen pair, then move forward one day.
- **Reward** = equal-weight next-day portfolio return minus **0.10% per stock replaced** (turnover cost). `1.0` means +1%.
- **Episode** = the full 30-day game.

In [ ]:
import itertools
import numpy as np

SEED = 7
rng = np.random.default_rng(SEED)
STOCKS = np.array(['ALFA', 'BETA', 'CORA', 'DUNE', 'ECHO', 'FARO'])
DATES = np.arange(np.datetime64('2026-07-06'), np.datetime64('2026-08-05'))
PORTFOLIOS = list(itertools.combinations(range(len(STOCKS)), 2))

# One known, reproducible market path: different trends plus daily noise.
days = np.arange(len(DATES) - 1)
base_drift = np.array([0.0030, 0.0015, -0.0005, 0.0022, -0.0018, 0.0007])
seasonality = 0.002 * np.sin(days[:, None] / 3 + np.arange(len(STOCKS)))
daily_returns = base_drift + seasonality + rng.normal(0, 0.006, size=(len(days), len(STOCKS)))
prices = 100 * np.vstack([np.ones(len(STOCKS)), np.cumprod(1 + daily_returns, axis=0)])

print(f'Dates: {DATES[0]} to {DATES[-1]} ({len(DATES)} calendar days)')
print('Stocks:', ', '.join(STOCKS))
print(f'Legal actions: C(6, 2) = {len(PORTFOLIOS)} portfolios')
print('First five actions:', [(i, tuple(STOCKS[list(p)])) for i, p in enumerate(PORTFOLIOS[:5])])

In [ ]:
class SixStockAssetEnv:
    """Minimal Gym-style environment with reset() and step(action)."""
    def __init__(self, returns, portfolios, start_pair=(0, 1), turnover_cost=0.10):
        self.returns = np.asarray(returns)
        self.portfolios = list(portfolios)
        self.turnover_cost = turnover_cost  # percentage points per replaced stock
        self.n_actions = len(self.portfolios)
        self.n_days = len(self.returns)
        self.start_action = self.portfolios.index(start_pair)
        self.n_states = self.n_days * self.n_actions

    def _state_id(self):
        return self.day * self.n_actions + self.holding_id

    def reset(self):
        self.day, self.holding_id = 0, self.start_action
        return self._state_id(), {'day': str(DATES[self.day]), 'holdings': self.portfolios[self.holding_id]}

    def step(self, action):
        if not 0 <= int(action) < self.n_actions:
            raise ValueError(f'action must be in [0, {self.n_actions - 1}]')
        action = int(action)
        old, new = set(self.portfolios[self.holding_id]), set(self.portfolios[action])
        replaced = len(old - new)
        gross_return_pct = 100 * self.returns[self.day, list(self.portfolios[action])].mean()
        reward = gross_return_pct - self.turnover_cost * replaced
        self.holding_id, self.day = action, self.day + 1
        terminated = self.day == self.n_days
        next_state = None if terminated else self._state_id()
        info = {'gross_return_pct': gross_return_pct, 'turnover_cost_pct': self.turnover_cost * replaced,
                'holdings': self.portfolios[action], 'date': str(DATES[self.day])}
        return next_state, reward, terminated, info

env = SixStockAssetEnv(daily_returns, PORTFOLIOS)
state, info = env.reset()
print('states:', env.n_states, '= 29 trading decisions × 15 possible holdings')
print('initial state:', state, info)

## Environment sanity check

A legal action always maps to exactly two holdings. The reward components are exposed separately to make the environment easy to inspect.

In [ ]:
state, _ = env.reset()
next_state, reward, done, info = env.step(0)  # ALFA+BETA: no initial turnover
assert len(info['holdings']) == 2
assert not done and next_state is not None
assert np.isclose(reward, info['gross_return_pct'] - info['turnover_cost_pct'])
print('action 0 holdings:', tuple(STOCKS[list(info['holdings'])]))
print(f"reward = {info['gross_return_pct']:.3f}% gross - {info['turnover_cost_pct']:.3f}% turnover = {reward:.3f}%")
print('environment checks passed')

## Learn a policy with Q-learning

The Q-table has one row per state and one column per legal two-stock action. In this small fixed-path game it can learn the Bellman update end to end.

In [ ]:
def train_q_learning(env, episodes=12_000, alpha=0.18, gamma=0.97, epsilon_start=1.0, epsilon_end=0.02, seed=1):
    rng = np.random.default_rng(seed)
    q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0.0
        epsilon = epsilon_start + (epsilon_end - epsilon_start) * episode / (episodes - 1)
        done = False
        while not done:
            action = int(rng.integers(env.n_actions)) if rng.random() < epsilon else int(np.argmax(q[state]))
            next_state, reward, done, _ = env.step(action)
            target = reward if done else reward + gamma * np.max(q[next_state])
            q[state, action] += alpha * (target - q[state, action])
            state, total_reward = next_state, total_reward + reward
        returns.append(total_reward)
    return q, np.array(returns)

q_table, training_returns = train_q_learning(env)
print('Q-table shape:', q_table.shape)
print(f'average reward, first 500 episodes: {training_returns[:500].mean():.2f}%')
print(f'average reward, last 500 episodes:  {training_returns[-500:].mean():.2f}%')

In [ ]:
def play_greedy_policy(env, q):
    state, _ = env.reset()
    history, done = [], False
    while not done:
        action = int(np.argmax(q[state]))
        next_state, reward, done, info = env.step(action)
        history.append((info['date'], tuple(STOCKS[list(info['holdings'])]), reward))
        state = next_state
    return history

history = play_greedy_policy(env, q_table)
profit_pct = sum(row[2] for row in history)
print(f'Greedy policy total reward / approximate profit: {profit_pct:.2f}%')
print('First 8 decisions:')
for date, holdings, reward in history[:8]:
    print(f'  {date}: {holdings} | reward {reward:+.3f}%')
print('All decisions hold exactly two stocks:', all(len(holdings) == 2 for _, holdings, _ in history))

## Takeaway

The table has `29 × 15 = 435` state-action values. Real markets need observations that do not reveal future returns (price history, indicators, and risk limits) plus out-of-sample evaluation. Here the focus is the MDP mechanics: legal portfolio actions, transparent rewards, and a complete training loop.

## $1,000 portfolio explorer

Choose any legal two-stock portfolio. The chart compounds its daily equal-weight returns from an initial **$1,000**; it does not include the RL turnover cost, so it is a clean comparison of each pair's market performance.

In [ ]:
import itertools
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, display

# Keep this explorer runnable by itself, even if earlier notebook cells were not run.
if 'PORTFOLIOS' not in globals() or 'daily_returns' not in globals():
    STOCKS = np.array(['ALFA', 'BETA', 'CORA', 'DUNE', 'ECHO', 'FARO'])
    DATES = np.arange(np.datetime64('2026-07-06'), np.datetime64('2026-08-05'))
    PORTFOLIOS = list(itertools.combinations(range(len(STOCKS)), 2))
    local_rng = np.random.default_rng(7)
    days = np.arange(len(DATES) - 1)
    base_drift = np.array([0.0030, 0.0015, -0.0005, 0.0022, -0.0018, 0.0007])
    seasonality = 0.002 * np.sin(days[:, None] / 3 + np.arange(len(STOCKS)))
    daily_returns = base_drift + seasonality + local_rng.normal(0, 0.006, size=(len(days), len(STOCKS)))

STARTING_CASH = 1_000.0
dropdown = widgets.Dropdown(
    options=[(f'{STOCKS[a]} + {STOCKS[b]}', i) for i, (a, b) in enumerate(PORTFOLIOS)],
    value=0, description='Portfolio:', style={'description_width': 'initial'}, layout=widgets.Layout(width='260px')
)
chart_output = widgets.Output()

def portfolio_chart(action_id):
    pair = PORTFOLIOS[action_id]
    values = STARTING_CASH * np.cumprod(np.r_[1.0, 1 + daily_returns[:, list(pair)].mean(axis=1)])
    # Wide, scrollable SVG: 30 daily dates and values stay readable.
    width, height, left, right, top, bottom = 1_600, 460, 58, 20, 45, 115
    lo, hi = values.min(), values.max()
    pad = max(8, (hi - lo) * 0.12)
    lo, hi = lo - pad, hi + pad
    x = np.linspace(left, width - right, len(values))
    y = top + (hi - values) / (hi - lo) * (height - top - bottom)
    points = ' '.join(f'{xi:.1f},{yi:.1f}' for xi, yi in zip(x, y))
    gain = values[-1] - STARTING_CASH
    color = '#14804a' if gain >= 0 else '#c9372c'
    date_labels = ''.join(f'<text x="{xi:.1f}" y="{height-22}" font-size="12" fill="white" text-anchor="end" transform="rotate(-55 {xi:.1f} {height-22})">{str(date)[8:10]} {str(date)[5:7]}</text>' for xi, date in zip(x, DATES))
    value_labels = ''.join(f'<text x="{xi:.1f}" y="{max(top + 12, yi - 9):.1f}" font-size="11" text-anchor="middle" fill="{color}">${value:,.0f}</text>' for xi, yi, value in zip(x, y, values))
    dots = ''.join(f'<circle cx="{xi:.1f}" cy="{yi:.1f}" r="3.5" fill="{color}"/>' for xi, yi in zip(x, y))
    return f'''<div style="max-width:100%;overflow-x:auto;font-family:system-ui,sans-serif">
      <div style="font-size:18px;font-weight:600">{STOCKS[pair[0]]} + {STOCKS[pair[1]]}</div>
      <div style="margin:4px 0 10px">$1,000 → <b style="color:{color}">${values[-1]:,.2f}</b> &nbsp; ({gain / STARTING_CASH:+.2%})</div>
      <svg viewBox="0 0 {width} {height}" width="{width}" height="{height}" role="img" aria-label="Daily portfolio value over time">
        <rect x="{left}" y="{top}" width="{width-left-right}" height="{height-top-bottom}" fill="none" stroke="#9aa0a6"/>
        <line x1="{left}" x2="{width-right}" y1="{top + (hi-1000)/(hi-lo)*(height-top-bottom):.1f}" y2="{top + (hi-1000)/(hi-lo)*(height-top-bottom):.1f}" stroke="#9aa0a6" stroke-dasharray="4 4"/>
        <polyline points="{points}" fill="none" stroke="{color}" stroke-width="3"/>
        {dots}
        {value_labels}
        <text x="4" y="{top+8}" font-size="12">${hi:,.0f}</text><text x="4" y="{height-bottom}" font-size="12">${lo:,.0f}</text>
        {date_labels}
      </svg></div>'''

def update_chart(change=None):
    with chart_output:
        chart_output.clear_output(wait=True)
        display(HTML(portfolio_chart(dropdown.value)))

# VS Code sometimes renders ipywidgets only as VBox(...). This plain HTML/SVG
# fallback always shows the chart and keeps a browser-native dropdown.
import json
option_html = ''.join(f'<option value="{i}">{STOCKS[a]} + {STOCKS[b]}</option>' for i, (a, b) in enumerate(PORTFOLIOS))
all_charts = json.dumps([portfolio_chart(i) for i in range(len(PORTFOLIOS))])
explorer_html = f'''<div id="portfolio-explorer" style="max-width:720px">
<label for="portfolio-select"><b>Portfolio: </b></label><select id="portfolio-select">{option_html}</select>
<div id="portfolio-chart">{portfolio_chart(0)}</div>
</div><script>
(() => {{ const root = document.getElementById('portfolio-explorer');
const charts = {all_charts}; const select = root.querySelector('#portfolio-select');
const target = root.querySelector('#portfolio-chart');
select.addEventListener('change', () => {{ target.innerHTML = charts[Number(select.value)]; }});
}})();</script>'''
display(HTML(explorer_html))

## Benchmark comparison

We compare the learned agent against a **Buy & Hold ALFA + BETA** portfolio (the same two stocks it starts with) and a six-stock equal-weight **market benchmark**. All three begin with $1,000. The agent curve includes its turnover cost; the passive benchmarks do not trade after day one.

In [ ]:
def capital_curve_from_rewards(rewards, starting_cash=1_000.0):
    return starting_cash * np.cumprod(np.r_[1.0, 1 + np.asarray(rewards) / 100])

rl_curve = capital_curve_from_rewards([reward for _, _, reward in history])
buy_hold_curve = STARTING_CASH * np.cumprod(np.r_[1.0, 1 + daily_returns[:, [0, 1]].mean(axis=1)])
market_curve = STARTING_CASH * np.cumprod(np.r_[1.0, 1 + daily_returns.mean(axis=1)])
series = [('RL agent (after turnover)', rl_curve, '#2563eb'), ('Buy & Hold: ALFA + BETA', buy_hold_curve, '#16a34a'), ('Market: equal weight 6 stocks', market_curve, '#f59e0b')]

width, height, left, right, top, bottom = 1040, 440, 70, 20, 42, 90
all_values = np.concatenate([values for _, values, _ in series])
lo, hi = all_values.min(), all_values.max()
pad = max(12, (hi - lo) * 0.10)
lo, hi = lo - pad, hi + pad
x = np.linspace(left, width - right, len(DATES))
def line_points(values):
    y = top + (hi - values) / (hi - lo) * (height - top - bottom)
    return ' '.join(f'{xi:.1f},{yi:.1f}' for xi, yi in zip(x, y))
paths = ''.join(f'<polyline points="{line_points(values)}" fill="none" stroke="{color}" stroke-width="3"/>' for _, values, color in series)
legend = ''.join(f'<span style="color:{color};margin-right:18px">● {name}: ${values[-1]:,.2f}</span>' for name, values, color in series)
date_labels = ''.join(f'<text x="{xi:.1f}" y="{height-22}" font-size="11" fill="white" text-anchor="end" transform="rotate(-55 {xi:.1f} {height-22})">{str(date)[8:10]} {str(date)[5:7]}</text>' for xi, date in zip(x, DATES))
benchmark_html = f'''<div style="max-width:100%;overflow-x:auto;font-family:system-ui,sans-serif">
<div style="margin:0 0 8px">{legend}</div>
<svg viewBox="0 0 {width} {height}" width="{width}" height="{height}" role="img" aria-label="RL agent and benchmark capital comparison">
<rect x="{left}" y="{top}" width="{width-left-right}" height="{height-top-bottom}" fill="none" stroke="#9aa0a6"/>
<line x1="{left}" x2="{width-right}" y1="{top + (hi-1000)/(hi-lo)*(height-top-bottom):.1f}" y2="{top + (hi-1000)/(hi-lo)*(height-top-bottom):.1f}" stroke="#9aa0a6" stroke-dasharray="4 4"/>
{paths}<text x="6" y="{top+8}" font-size="12">${hi:,.0f}</text><text x="6" y="{height-bottom}" font-size="12">${lo:,.0f}</text>{date_labels}
</svg></div>'''
display(HTML(benchmark_html))
for name, values, _ in series:
    print(f'{name}: ${values[-1]:,.2f} ({values[-1] / STARTING_CASH - 1:+.2%})')

# Play the Asset Management Game

You are the player. Start with **$1,000**, choose a portfolio of exactly **two stocks**, then press **Next day**. Only prices up to the current day are shown—future prices stay hidden until you advance. Replacing a stock costs 0.10% per stock, exactly like the RL environment.

In [ ]:
# Browser-native controls are used so the game also works in notebook UIs that do not render ipywidgets.
game_prices = json.dumps(prices.round(2).tolist())
game_dates = json.dumps([str(d) for d in DATES])
game_stocks = json.dumps(STOCKS.tolist())
game_pairs = json.dumps([list(pair) for pair in PORTFOLIOS])
game_html = f'''<div id="asset-game" style="font-family:system-ui,sans-serif;max-width:980px">
  <div style="display:flex;gap:22px;flex-wrap:wrap;align-items:end;margin-bottom:12px">
    <div><div style="font-size:13px">Current date</div><b id="game-date"></b></div>
    <div><div style="font-size:13px">Your capital</div><b id="game-cash"></b></div>
    <div><label for="game-pair">Choose exactly 2 stocks</label><br><select id="game-pair"></select></div>
    <button type="button" id="next-day">Next day</button><button type="button" id="restart-game">Restart</button>
  </div><div id="game-message" style="margin-bottom:12px"></div>
  <div id="stock-cards" style="display:grid;grid-template-columns:repeat(3,minmax(230px,1fr));gap:10px"></div>
  <div id="game-history" style="margin-top:14px"></div>
</div><script>
(() => {{
  const root = document.getElementById('asset-game');
  const prices = {game_prices}, dates = {game_dates}, stocks = {game_stocks}, pairs = {game_pairs};
  const select = root.querySelector('#game-pair'), next = root.querySelector('#next-day');
  pairs.forEach((p,i) => select.add(new Option(`${{stocks[p[0]]}} + ${{stocks[p[1]]}}`, i)));
  let day, cash, held, moves;
  const money = n => `$${{n.toLocaleString(undefined, {{minimumFractionDigits:2, maximumFractionDigits:2}})}}`;
  function spark(values, color) {{
    const w=270,h=100,p=12, lo=Math.min(...values), hi=Math.max(...values), range=Math.max(hi-lo, 1);
    const pts=values.map((v,i)=>`${{p+i*(w-2*p)/(values.length-1||1)}},${{p+(hi-v)/range*(h-2*p)}}`).join(' ');
    return `<svg viewBox="0 0 ${{w}} ${{h}}" width="100%" height="100"><polyline points="${{pts}}" fill="none" stroke="${{color}}" stroke-width="2.5"/></svg>`;
  }}
  function render() {{
    root.querySelector('#game-date').textContent = dates[day];
    root.querySelector('#game-cash').textContent = money(cash);
    root.querySelector('#stock-cards').innerHTML = stocks.map((name,i) => {{
      const now=prices[day][i], before=prices[Math.max(0,day-1)][i], change=day ? (now/before-1)*100 : 0;
      return `<div style="padding:10px;border:1px solid #888"><b>${{name}}</b> &nbsp; $${{now.toFixed(2)}} <span style="color:${{change>=0?'#14804a':'#c9372c'}}">${{change>=0?'+':''}}${{change.toFixed(2)}}%</span>${{spark(prices.slice(0,day+1).map(row=>row[i]), change>=0?'#16a34a':'#dc2626')}}</div>`;
    }}).join('');
    const holdingText = held === null ? 'Pick your first two stocks, then reveal tomorrow.' : `Holding: ${{stocks[held[0]]}} + ${{stocks[held[1]]}}`;
    root.querySelector('#game-message').textContent = day === dates.length-1 ? `Game over! Final capital: ${{money(cash)}} (${{((cash/1000-1)*100).toFixed(2)}}%).` : holdingText;
    next.disabled = day === dates.length-1; select.disabled = day === dates.length-1;
    root.querySelector('#game-history').innerHTML = moves.length ? '<b>Decisions</b><br>' + moves.map(m => `${{m.date}} — ${{m.pair}}: ${{m.reward>=0?'+':''}}${{m.reward.toFixed(2)}}%, capital ${{money(m.cash)}}`).join('<br>') : '';
  }}
  function advance() {{
    if (day >= dates.length-1) return;
    const pair=pairs[Number(select.value)], replaced=held===null ? 0 : held.filter(x=>!pair.includes(x)).length;
    const gross=((prices[day+1][pair[0]]/prices[day][pair[0]]-1)+(prices[day+1][pair[1]]/prices[day][pair[1]]-1))/2*100;
    const reward=gross-0.10*replaced; cash*=1+reward/100; held=pair; day++;
    moves.push({{date:dates[day], pair:`${{stocks[pair[0]]}} + ${{stocks[pair[1]]}}`, reward, cash}}); render();
  }}
  function restart() {{ day=0; cash=1000; held=null; moves=[]; select.value=0; render(); }}
  next.addEventListener('click',advance); root.querySelector('#restart-game').addEventListener('click',restart); restart();
}})();</script>'''
display(HTML(game_html))